# Manipulation planning: notebook

The 2R arm of RO-036 (links 0.3 m and 0.2 m), a cup, a bottle and a table, side view (`images/manip.py`); the
cup holder and the motion "move down" (`images/holder.py`). Here we compute every number of the Note. Seeds are fixed.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import shapely
import manip as m
import holder as H

## 1. The scene and the opening problem (Note, Section 1)

In [2]:
print('cup', m.CUP_W, 'm wide,', m.CUP_H, 'm tall; spots A and B at x =', m.A_X, 'and', m.B_X, 'm, apart by', round(m.B_X - m.A_X, 2))
print('bottle from x =', m.BOTTLE[0], 'to', m.BOTTLE[1], 'm, height', round(m.BOTTLE[3] - m.TOP, 2), 'm')
GA = [s for s in m.ik(m.lid(m.A_X)) if not m.collide(*s, 'transit A')[0]][0]
GB = [s for s in m.ik(m.lid(m.B_X)) if not m.collide(*s, 'transit B')[0]][0]
print('grasp at A (hand on the lid):', np.round(GA, 1), ' place at B:', np.round(GB, 1), ' home:', m.HOME)
for name, q in (('grasp', GA), ('place', GB)):
    print(f'  {name}: elbow and hand', [np.round(v, 3) for v in m.fk(*q)])

cup 0.06 m wide, 0.08 m tall; spots A and B at x = 0.18 and 0.42 m, apart by 0.24
bottle from x = 0.31 to 0.35 m, height 0.14 m
grasp at A (hand on the lid): [  34.  -144.1]  place at B: [ 23.2 -67. ]  home: (90.0, 0.0)
  grasp: elbow and hand [array([0.249, 0.168]), array([ 0.18, -0.02])]
  place: elbow and hand [array([0.276, 0.118]), array([ 0.42, -0.02])]


In [3]:
route, L, V, E = m.prm_plan('arm', GA, GB, n=150, k=10, seed=0)
P = np.array(m.densify(route, 0.05))
(ex, ey), (hx, hy) = m.fk(P[:, 0], P[:, 1])
cups = shapely.box(hx - m.CUP_W / 2, hy - m.CUP_H, hx + m.CUP_W / 2, hy)
print('route for the arm alone:', round(L, 1), 'degrees;', len(route), 'configurations; arm collides anywhere:',
      bool(m.collide(P[:, 0], P[:, 1], 'arm').any()))
print('share of the route with the cup in the table', round(shapely.intersects(m.floor().buffer(-m.TOUCH), cups).mean(), 3),
      '; in the bottle', round(shapely.intersects(m.bottle(), cups).mean(), 3),
      '; deepest in the table', round(float((m.TOP - (hy - m.CUP_H)).max()), 4), 'm')

route for the arm alone: 109.3 degrees; 5 configurations; arm collides anywhere: False
share of the route with the cup in the table 0.143 ; in the bottle 0.322 ; deepest in the table 0.0172 m


## 2. Transit and transfer (Note, Section 2)

In [4]:
for mode in ('transit A', 'transfer', 'transit B'):
    g, C = m.cause_grid(mode, 1.0)
    print(f'{mode:10s}: free {(C == 0).mean():.3f}; hits table or robot {(C == 1).mean():.3f}, bottle {(C == 2).mean():.3f}, '
          f'cup {(C == 3).mean():.3f}')
for name, q in (('grasp', GA), ('place', GB), ('home', m.HOME)):
    print(f'{name}: free in transit A {not m.collide(*q, "transit A")[0]}, transfer {not m.collide(*q, "transfer")[0]}, '
          f'transit B {not m.collide(*q, "transit B")[0]}')
print('grasp: distance from link 2 to the cup at A:',
      round(shapely.LineString([m.fk(*GA)[0], m.fk(*GA)[1]]).distance(m.cup_at(m.A_X)), 6), 'm (touching)')

transit A : free 0.494; hits table or robot 0.384, bottle 0.028, cup 0.094


transfer  : free 0.423; hits table or robot 0.474, bottle 0.058, cup 0.045
transit B : free 0.513; hits table or robot 0.436, bottle 0.035, cup 0.016
grasp: free in transit A True, transfer True, transit B True
place: free in transit A True, transfer True, transit B True
home: free in transit A True, transfer True, transit B True
grasp: distance from link 2 to the cup at A: 0.0 m (touching)


In [5]:
legs = [('transit A', m.HOME, GA), ('transfer', GA, GB), ('transit B', GB, m.HOME)]
total = 0
for mode, s, g in legs:
    Pl, Ll, Vl, El = m.prm_plan(mode, s, g, n=150, k=10, seed=0)
    total += Ll
    D = np.array(m.densify(Pl, 0.5))
    print(f'{mode:10s}: {len(Vl) - 2} milestones, {len(El)} edges, route {Ll:.1f} degrees through {len(Pl) - 2} milestones, '
          f'free {not m.collide(D[:, 0], D[:, 1], mode).any()}')
print('whole task', round(total, 1), 'degrees')
Pt, Lt, _, _ = m.prm_plan('transfer', GA, GB, n=150, k=10, seed=0)
D = np.array(m.densify(Pt, 0.5))
(ex, ey), (hx, hy) = m.fk(D[:, 0], D[:, 1])
print('transfer: highest point of the cup bottom', round(float((hy - m.CUP_H).max()), 3), 'm; bottle top', m.BOTTLE[3])

transit A : 150 milestones, 871 edges, route 169.0 degrees through 5 milestones, free True


transfer  : 150 milestones, 859 edges, route 123.2 degrees through 3 milestones, free True


transit B : 150 milestones, 875 edges, route 99.2 degrees through 3 milestones, free True
whole task 391.5 degrees


transfer: highest point of the cup bottom 0.076 m; bottle top 0.04


## 3. Pushing and the quasistatic assumption (Note, Section 3)

In [6]:
mass, g, mu, w = 0.25, 9.81, 0.4, m.CUP_W
weight = mass * g
friction = mu * weight
print('weight', round(weight, 4), 'N; friction at most', round(friction, 4), 'N')
a = 0.05 / 0.5
print('speeding up to 0.05 m/s in 0.5 s: a =', a, 'm/s2, m a =', round(mass * a, 4), 'N =', round(mass * a / friction, 4), 'of friction')
dec = mu * g
for v in (0.05, 0.5):
    t = v / dec
    print(f'coasting from {v} m/s: slowed by {dec:.3f} m/s2, stops after {t:.4f} s and {v / 2 * t * 1000:.2f} mm')
for p in (0.04, 0.08):
    print(f'push at {p} m: push torque about the front corner {friction * p:.4f} N m; weight torque {weight * w / 2:.4f} N m')
print('highest push that slides: w / (2 mu) =', round(w / (2 * mu), 4), 'm')

weight 2.4525 N; friction at most 0.981 N
speeding up to 0.05 m/s in 0.5 s: a = 0.1 m/s2, m a = 0.025 N = 0.0255 of friction
coasting from 0.05 m/s: slowed by 3.924 m/s2, stops after 0.0127 s and 0.32 mm
coasting from 0.5 m/s: slowed by 3.924 m/s2, stops after 0.1274 s and 31.86 mm
push at 0.04 m: push torque about the front corner 0.0392 N m; weight torque 0.0736 N m
push at 0.08 m: push torque about the front corner 0.0785 N m; weight torque 0.0736 N m
highest push that slides: w / (2 mu) = 0.075 m


## 4. Preimage planning (Note, Section 4)

In [7]:
print('friction angle: atan(0.4) =', round(H.FRICTION_ANGLE, 2), 'degrees')
print('slot', round(2 * H.GOAL_HALF, 3), 'm wide; mouth', round(2 * (H.GOAL_HALF + H.CHAMFER), 3), 'm wide')
print('chamfer: a motion within 10 degrees of down is 35 to 55 degrees away from straight into the chamfer')
xs, ys, G0 = H.preimage_grid(False, h=0.001)
xs, ys, G1 = H.preimage_grid(True, h=0.001)
for name, Gs in (('without sliding', G0), ('with sliding', G1)):
    yy = ys[Gs.any(1)]
    j = np.argmin(abs(ys - H.MOUTH))
    row = xs[Gs[j]]
    print(f'{name}: top {yy.max() - H.MOUTH:.3f} m above the posts; at the posts\' top {row.min() - H.B:.3f} to '
          f'{row.max() - H.B:.3f} m from the slot centre')
tan10 = np.tan(np.radians(10))
print('tan 10 degrees =', round(tan10, 4))
print('drift over 2 cm of fall:', round(0.02 * tan10, 4), 'm')
print('band of +-1 cm fits up to', round((0.015 - 0.01) / tan10, 4), 'm above the posts')
for h in (0.014, 0.05):
    j = np.argmin(abs(ys - (H.MOUTH + h)))
    row = xs[G1[j]]
    print(f'at {h} m above the posts the preimage spans {row.min() - H.B:+.3f} to {row.max() - H.B:+.3f} m')
S = (H.B + 0.008, H.MOUTH + 0.02)
for slide in (False, True):
    print('start 0.8 cm right, 2 cm up, slide', slide, [bool(H.run(S, ang, slide)[1]) for ang in (260, 265, 270, 275, 280)])

friction angle: atan(0.4) = 21.8 degrees
slot 0.01 m wide; mouth 0.03 m wide
chamfer: a motion within 10 degrees of down is 35 to 55 degrees away from straight into the chamfer


without sliding: top 0.008 m above the posts; at the posts' top -0.001 to 0.001 m from the slot centre
with sliding: top 0.085 m above the posts; at the posts' top -0.014 to 0.014 m from the slot centre
tan 10 degrees = 0.1763
drift over 2 cm of fall: 0.0035 m
band of +-1 cm fits up to 0.0284 m above the posts
at 0.014 m above the posts the preimage spans -0.012 to +0.012 m
at 0.05 m above the posts the preimage spans -0.006 to +0.006 m
start 0.8 cm right, 2 cm up, slide False [True, False, False, False, False]
start 0.8 cm right, 2 cm up, slide True [True, True, True, True, True]
